In [2]:
import pandas as pd
import numpy as np

from sklearn.model_selection import KFold
from sklearn.metrics import mean_absolute_percentage_error

from sklearn.preprocessing import OrdinalEncoder
from sklearn.impute import SimpleImputer

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD

from catboost import CatBoostRegressor
from lightgbm import LGBMRegressor

import warnings
warnings.filterwarnings('ignore')

train = pd.read_csv('train.csv')
test = pd.read_csv('test_x.csv')

TARGET = 'salary_mean_net'
ID_COL = 'id'

y = train[TARGET]
y_log = np.log1p(y)

train_features = train.drop(columns=[TARGET])

full = pd.concat(
    [train_features, test],
    axis=0
).reset_index(drop=True)

full = full.replace([
    'не указано',
    'Не указано',
    'не заполнено'
], np.nan)

In [3]:
text_cols = [
    'name',
    'employer_name',
    'key_skills_name',
    'specializations_profarea_name',
    'professional_roles_name',
    'raw_description',
    'raw_branded_description',
    'lemmaized_wo_stopwords_raw_description',
    'lemmaized_wo_stopwords_raw_branded_description'
]

for col in text_cols:
    if col not in full.columns:
        full[col] = ''

    full[col] = full[col].fillna('').astype(str)

full['all_text'] = (
    full['name'] + ' ' +
    full['employer_name'] + ' ' +
    full['professional_roles_name'] + ' ' +
    full['specializations_profarea_name'] + ' ' +
    full['lemmaized_wo_stopwords_raw_description'] + ' ' +
    full['lemmaized_wo_stopwords_raw_branded_description']
)

In [4]:
full['desc_len'] = (
    full['raw_description']
    .fillna('')
    .astype(str)
    .str.len()
)

full['desc_words'] = (
    full['raw_description']
    .fillna('')
    .astype(str)
    .apply(lambda x: len(x.split()))
)

full['brand_words'] = (
    full['raw_branded_description']
    .fillna('')
    .astype(str)
    .apply(lambda x: len(x.split()))
)

full['skills_count'] = (
    full['key_skills_name']
    .fillna('')
    .astype(str)
    .apply(lambda x: len(x.split(',')))
)

full['uppercase_ratio'] = (
    full['raw_description']
    .fillna('')
    .astype(str)
    .apply(lambda x: sum(1 for c in x if c.isupper()) / (len(x) + 1))
)

full['digit_count'] = (
    full['raw_description']
    .fillna('')
    .astype(str)
    .apply(lambda x: sum(c.isdigit() for c in x))
)

full['exclamation_count'] = (
    full['raw_description']
    .fillna('')
    .astype(str)
    .str.count('!')
)

full['has_bonus'] = (
    full['all_text']
    .str.contains(
        'бонус|премия|процент|kpi|оклад',
        case=False,
        regex=True
    )
    .astype(int)
)

full['has_remote'] = (
    full['all_text']
    .str.contains(
        'удален|remote|гибрид',
        case=False,
        regex=True
    )
    .astype(int)
)

full['has_english'] = (
    full['all_text']
    .str.contains(
        'english|англий',
        case=False,
        regex=True
    )
    .astype(int)
)

full['is_moscow'] = (
    full['unified_address_city']
    .fillna('')
    .astype(str)
    .str.lower()
    .str.contains('москва')
).astype(int)

full['is_spb'] = (
    full['unified_address_city']
    .fillna('')
    .astype(str)
    .str.lower()
    .str.contains('санкт')
).astype(int)

In [5]:
high_cardinality = [
    'employer_name',
    'name_clean',
    'employer_industries',
    'professional_roles_name',
    'specializations_profarea_name'
]

for col in high_cardinality:

    if col not in train.columns:
        continue

    train_col = (
        train[col]
        .fillna('missing')
        .astype(str)
    )

    stats = (
        pd.DataFrame({
            col: train_col,
            'target': y
        })
        .groupby(col)['target']
        .agg(['mean', 'count'])
    )

    global_mean = y.mean()

    smooth = (
        (stats['mean'] * stats['count'] + global_mean * 30)
        / (stats['count'] + 30)
    )

    mapping = smooth.to_dict()

    full[f'{col}_target_enc'] = (
        full[col]
        .fillna('missing')
        .astype(str)
        .map(mapping)
        .fillna(global_mean)
    )

In [6]:
tfidf = TfidfVectorizer(
    max_features=120000,
    ngram_range=(1, 3),
    min_df=2,
    max_df=0.92,
    sublinear_tf=True,
    strip_accents='unicode'
)

text_matrix = tfidf.fit_transform(full['all_text'])

svd = TruncatedSVD(
    n_components=256,
    random_state=42
)

text_svd = svd.fit_transform(text_matrix)

text_svd = pd.DataFrame(
    text_svd,
    columns=[f'svd_{i}' for i in range(text_svd.shape[1])]
)

full = pd.concat(
    [full.reset_index(drop=True), text_svd],
    axis=1
)

In [7]:
drop_cols = [
    'all_text',
    'raw_description',
    'raw_branded_description',
    'lemmaized_wo_stopwords_raw_description',
    'lemmaized_wo_stopwords_raw_branded_description',
    'name',
    'employer_name',
    'key_skills_name'
]

existing_drop = [
    c for c in drop_cols
    if c in full.columns
]

full = full.drop(columns=existing_drop)

obj_cols = full.select_dtypes(include='object').columns

for col in obj_cols:

    full[col] = (
        full[col]
        .fillna('missing')
        .astype(str)
    )

    encoder = OrdinalEncoder(
        handle_unknown='use_encoded_value',
        unknown_value=-1
    )

    full[[col]] = encoder.fit_transform(full[[col]])

train_final = full.iloc[:len(train)].copy()
test_final = full.iloc[len(train):].copy()

test_ids = test_final[ID_COL]

train_final = train_final.drop(
    columns=[ID_COL],
    errors='ignore'
)

test_final = test_final.drop(
    columns=[ID_COL],
    errors='ignore'
)

train_final = train_final.replace(
    [np.inf, -np.inf],
    np.nan
)

test_final = test_final.replace(
    [np.inf, -np.inf],
    np.nan
)

imputer = SimpleImputer(strategy='median')

train_final = pd.DataFrame(
    imputer.fit_transform(train_final),
    columns=train_final.columns
)

test_final = pd.DataFrame(
    imputer.transform(test_final),
    columns=test_final.columns
)

print(train_final.shape)
print(test_final.shape)

(49051, 290)
(12263, 290)


In [8]:
kf = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

oof_cat = np.zeros(len(train_final))
oof_lgb = np.zeros(len(train_final))

test_preds_cat = np.zeros(len(test_final))
test_preds_lgb = np.zeros(len(test_final))

scores_cat = []
scores_lgb = []

In [ ]:
for fold, (train_idx, valid_idx) in enumerate(kf.split(train_final)):

    X_train = train_final.iloc[train_idx]
    X_valid = train_final.iloc[valid_idx]

    y_train = y_log.iloc[train_idx]
    y_valid = y.iloc[valid_idx]

    model = CatBoostRegressor(
        iterations=6000,
        learning_rate=0.02,
        depth=10,
        loss_function='RMSE',
        eval_metric='MAPE',
        random_seed=42,
        bagging_temperature=0.5,
        random_strength=1,
        l2_leaf_reg=5,
        subsample=0.8,
        verbose=300
    )

    model.fit(
        X_train,
        y_train,
        eval_set=(
            X_valid,
            np.log1p(y_valid)
        ),
        use_best_model=True
    )

    preds_valid = np.expm1(
        model.predict(X_valid)
    )

    preds_test = np.expm1(
        model.predict(test_final)
    )

    oof_cat[valid_idx] = preds_valid

    test_preds_cat += (
        preds_test / kf.n_splits
    )

    score = mean_absolute_percentage_error(
        y_valid,
        preds_valid
    )

    scores_cat.append(score)

    print(f'Cat Fold {fold + 1}: {score}')

print('CatBoost CV:', np.mean(scores_cat))

0:	learn: 0.0399354	test: 0.0397120	best: 0.0397120 (0)	total: 1.11s	remaining: 1h 51m 30s
300:	learn: 0.0129077	test: 0.0139358	best: 0.0139358 (300)	total: 5m 7s	remaining: 1h 36m 55s
600:	learn: 0.0112471	test: 0.0129997	best: 0.0129997 (600)	total: 10m 13s	remaining: 1h 31m 47s
900:	learn: 0.0099583	test: 0.0124340	best: 0.0124340 (900)	total: 15m 19s	remaining: 1h 26m 41s
1200:	learn: 0.0089468	test: 0.0120982	best: 0.0120982 (1200)	total: 20m 30s	remaining: 1h 21m 56s
1500:	learn: 0.0081411	test: 0.0118656	best: 0.0118656 (1500)	total: 25m 36s	remaining: 1h 16m 45s


In [ ]:
for fold, (train_idx, valid_idx) in enumerate(kf.split(train_final)):

    X_train = train_final.iloc[train_idx]
    X_valid = train_final.iloc[valid_idx]

    y_train = y_log.iloc[train_idx]
    y_valid = y.iloc[valid_idx]

    model = LGBMRegressor(
        n_estimators=7000,
        learning_rate=0.015,
        num_leaves=128,
        max_depth=-1,
        subsample=0.8,
        colsample_bytree=0.8,
        min_child_samples=20,
        reg_alpha=0.1,
        reg_lambda=0.1,
        random_state=42
    )

    model.fit(
        X_train,
        y_train,
        eval_set=[
            (
                X_valid,
                np.log1p(y_valid)
            )
        ],
        eval_metric='l1'
    )

    preds_valid = np.expm1(
        model.predict(X_valid)
    )

    preds_test = np.expm1(
        model.predict(test_final)
    )

    oof_lgb[valid_idx] = preds_valid

    test_preds_lgb += (
        preds_test / kf.n_splits
    )

    score = mean_absolute_percentage_error(
        y_valid,
        preds_valid
    )

    scores_lgb.append(score)

    print(f'LGB Fold {fold + 1}: {score}')

print('LGBM CV:', np.mean(scores_lgb))

In [ ]:
ensemble_oof = (
    0.75 * oof_cat +
    0.25 * oof_lgb
)

ensemble_score = mean_absolute_percentage_error(
    y,
    ensemble_oof
)

print('ENSEMBLE CV:', ensemble_score)

In [ ]:
final_pred = (
    0.75 * test_preds_cat +
    0.25 * test_preds_lgb
)

final_pred = np.clip(final_pred, 0, None)

In [ ]:
submission = pd.DataFrame({
    'id': test_ids,
    'salary_mean_net': final_pred
})

submission.to_csv(
    'submission.csv',
    index=False
)

display(submission.head())